In [1]:
import textgrid
import math
import cv2
import numpy as np
import os
from tqdm import tqdm

In [2]:
user_name = 'astitva'

# set paths
ALIGNER_ROOT = f'/mnt/users_scratch/{user_name}/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/ForcedAligner'
ASSETS_ROOT = f'/mnt/users_scratch/{user_name}/WORKSPACE/Fedora-DGX-Codebase/GENERATION/PresetGeneration/OUTPUT/for_PPT'
VIDEO_SAVE_DIR = f'/mnt/users_scratch/{user_name}/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/VIDEO_RESULTS'
FFMPEG_PATH = f'/mnt/users_scratch/{user_name}/WORKSPACE/ffmpeg/installation/bin'

In [3]:
# helper function for compositing assets
def composite(base, mouth, eyes, use_default_mouth=False, use_default_eyes=False):
    mask_mouth = mouth[:,:,3]/255
    mask_eyes = eyes[:,:,3]/255
    mask = mask_mouth + mask_eyes
    if use_default_eyes:
      mask = mask_mouth
    if use_default_mouth:
        mask = mask_eyes
    mask_im = np.repeat(mask[..., np.newaxis], 3, axis=2)
    mask_mouth_im = np.repeat(mask_mouth[..., np.newaxis], 3, axis=2)
    mask_eyes_im = np.repeat(mask_eyes[..., np.newaxis], 3, axis=2)
    composited = base*(1-mask_im) 
    if not use_default_eyes:
        composited += mask_eyes_im*eyes[:,:,:3]
    if not use_default_mouth:
        composited += mask_mouth_im*mouth[:,:,:3]
    return composited.astype('uint8')

# ANIMATION : MOUTH + EYES 

### TEXT-GRID OBTAINED USING FORCED ALIGNMENT

In [11]:
filename = 'ppt'

tg_path = f'{ALIGNER_ROOT}/outputs/{filename}.TextGrid'
tg = textgrid.TextGrid.fromFile(tg_path)

words = tg[0]
phonemes = tg[1]

words_phonemes=None

words_phonemes=[]
phoneme_idx = 0
for w in words:
    current_set = []
    time = w.duration()
    start = 0.0
    while(time!=start):
        ph = phonemes[phoneme_idx]
        start += ph.duration()
        current_set.append(ph)
        phoneme_idx += 1
    words_phonemes.append(current_set)

words, phonemes

(IntervalTier(words, [Interval(0.0, 0.09, None), Interval(0.09, 0.65, hello), Interval(0.65, 1.16, None), Interval(1.16, 1.44, thanks), Interval(1.44, 1.55, for), Interval(1.55, 1.92, joining), Interval(1.92, 2.06, me), Interval(2.06, 2.55, today), Interval(2.55, 3.28, None), Interval(3.28, 3.37, i), Interval(3.37, 3.54, may), Interval(3.54, 3.8, look), Interval(3.8, 3.98, like), Interval(3.98, 4.04, a), Interval(4.04, 4.67, drawing), Interval(4.67, 4.92, None), Interval(4.92, 5.03, but), Interval(5.03, 5.12, i), Interval(5.12, 5.31, have), Interval(5.31, 5.58, full), Interval(5.58, 5.87, range), Interval(5.87, 6.01, of), Interval(6.01, 6.81, emotions), Interval(6.81, 7.18, None), Interval(7.18, 7.28, i), Interval(7.28, 7.46, can), Interval(7.46, 7.6, be), Interval(7.6, 8.2, happy), Interval(8.2, 8.79, None), Interval(8.79, 8.87, i), Interval(8.87, 9.05, can), Interval(9.05, 9.17, be), Interval(9.17, 9.74, shocked), Interval(9.74, 9.89, None), Interval(9.89, 9.96, or), Interval(9.96, 1

### DEFINE PHONE-VISEME MAPPING

In [12]:
phoneme2viseme = {
    'AA':3,
    'AE':7, 
    'AH':7,
    'AO':8,
    'AW':8,
    'AY':4,
    'AX':3,
    'B':1, 
    'CH':12,
    'D':10,
    'DH':12,
    'EH':7,
    'ER':7,
    'EY':2,
    'F':15,
    'G':11,
    'HH':11,
    'IH':7,
    'IY':4,
    'JH':12,
    'K':11,
    'L':10,
    'M':1,
    'N':10,
    'NG':11,
    'OW':8,
    'OY':8,
    'P':1,
    'R':10,
    'S':12,
    'SH':12,
    'T':12,
    'TH':10,
    'UH':9,
    'UW':9,
    'UX':9,
    'V':15,
    'W':15,
    'Y':4,
    'Z':12,
    'ZH':12
}




# # banana
# phoneme2viseme = {
#     'AA':3,
#     'AE':7, 
#     'AH':16,
#     'AO':8,
#     'AW':8,
#     'AY':5,
#     'AX':5,
#     'B':1, 
#     'CH':12,
#     'D':1,
#     'DH':12,
#     'EH':7,
#     'ER':7,
#     'EY':2,
#     'F':15,
#     'G':11,
#     'HH':11,
#     'IH':4,
#     'IY':4,
#     'JH':12,
#     'K':11,
#     'L':14,
#     'M':1,
#     'N':14,
#     'NG':11,
#     'OW':8,
#     'OY':8,
#     'P':1,
#     'R':10,
#     'S':14,
#     'SH':14,
#     'T':1,
#     'TH':14,
#     'UH':9,
#     'UW':9,
#     'UX':9,
#     'V':15,
#     'W':9,
#     'Y':4,
#     'Z':12,
#     'ZH':12
# }

### DEFINE EYE-WORD MAPPING

In [13]:
# eye_word_mapping = {'mommy':3,'daddy':0,'grandma':6,'grandpa':7, 'hunt':5}
eye_word_mapping = {'nice':3,"look":2,'full':8,'happy':3, 'shocked':5, 'disappointed':7, 'cool':3}

### CREATE ANIMATION

In [14]:
# # characters = ['0a4a8a95ac934f4e8d7b58561f7913c9__1453198705448541']
# characters = sorted(os.listdir(ASSETS_ROOT+'/mouth/'))

# crop_face = False
# fps=120

# ext = 'mp4'

# suffix = ''
# if crop_face:
#     suffix = '_face'

# skip_start_frames = 0
# if filename=='babyshark':
#     skip_start_frames = 1063

# for character_id in tqdm(characters):

#     MOUTH_ROOT = f'{ASSETS_ROOT}/mouth/{character_id}'
#     EYES_ROOT = f'{ASSETS_ROOT}/eyes/{character_id}'
#     SAVE_ROOT = f'{VIDEO_SAVE_DIR}/{character_id}/'
#     os.makedirs(SAVE_ROOT, exist_ok=True)

#     asset_dict = {}
#     asset_dict['inpainted_mouth_only'] = cv2.imread(f'{MOUTH_ROOT}/metadata/inpainted_mouth_only.png')
#     asset_dict['inpainted_eyes_only'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_eyes_only.png')
#     asset_dict['inpainted_eyes_mouth'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_eyes_mouth.png')
#     asset_dict['inpainted_face_mouth_only'] = cv2.imread(f'{MOUTH_ROOT}/metadata/inpainted_face_mouth_only.png')
#     asset_dict['inpainted_face_eyes_only'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_face_eyes_only.png')
#     asset_dict['inpainted_face_eyes_mouth'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_face_eyes_mouth.png')

#     w0, h0, _ = cv2.imread(f'{MOUTH_ROOT}/assets/mouth_0.png').shape
#     print(w0, h0)
    
#     mouth_files = os.listdir(f'{MOUTH_ROOT}/assets/')
#     for m in mouth_files:
#         asset_dict[m[:-4]] = cv2.imread(f'{MOUTH_ROOT}/assets/{m}', -1)

#     eye_files = os.listdir(f'{EYES_ROOT}/assets/')
#     for e in eye_files:
#         asset_dict[e[:-4]] = cv2.imread(f'{EYES_ROOT}/assets/{e}', -1)

#     asset_dict.keys()
    
#     base = asset_dict[f'inpainted{suffix}_mouth_only']
#     base = cv2.resize(base, (h0,w0))

#     eyes_id = 0
#     blink_id = 2
#     blink_gap = 2 #seconds
#     num_blink_frames = 12
    
#     USE_DEFAULT_MOUTH = False
#     USE_DEFAULT_EYES = True
    
#     video=cv2.VideoWriter(f'{SAVE_ROOT}/{filename}_no_audio.{ext}',cv2.VideoWriter_fourcc(*'mp4v'),fps,(h0,w0))
#     buffer = np.ones((w0,h0,3)).astype('uint8')*255
#     current_frame_count = 0
    
#     for i in tqdm(range(len(words))):
#         total_duration = words[i].duration()
#         word_frame_count = int(fps*total_duration)
#         cumulative_frame_count = 0
    
#         #eyes asset
#         try:
#             eyes_id = eye_word_mapping[words[i].mark.lower()]
#             base = asset_dict[f'inpainted{suffix}_eyes_mouth']
#             base = cv2.resize(base, (h0,w0))
#             USE_DEFAULT_EYES=False
#         except:
#             pass
    
#         eyes = asset_dict[f'eyes_{eyes_id}{suffix}']
#         eyes = cv2.resize(eyes, (h0,w0))


        
#         for p in words_phonemes[i]:
#             frame_count = math.ceil(fps*p.duration())
    
#             #default mouth asset
#             mouth_id = 1
#             buffer_mouth = asset_dict[f'mouth_{mouth_id}{suffix}']
#             buffer_mouth = cv2.resize(buffer_mouth, (h0,w0))

            
#             # default
#             buffer = composite(base, buffer_mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
            
#             viseme_id=''
#             if p.mark!='':
#                 try:
#                     mouth_id = phoneme2viseme[p.mark[:2]]
#                     mouth = asset_dict[f'mouth_{mouth_id}{suffix}']
#                     mouth = cv2.resize(mouth, (h0,w0))
#                     buffer_mouth = mouth
#                     buffer = composite(base, mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
#                 except:
#                     pass

#             for _ in range(frame_count):
#                 if cumulative_frame_count>=word_frame_count:
#                     break
#                 #blink
#                 if current_frame_count%(fps*blink_gap)<num_blink_frames:
#                     blink_eyes = asset_dict[f'eyes_{blink_id}{suffix}']
#                     blink_eyes = cv2.resize(blink_eyes, (h0,w0))
#                     blink_base = asset_dict[f'inpainted{suffix}_eyes_mouth']
#                     blink_base = cv2.resize(blink_base, (h0,w0))
#                     if current_frame_count>skip_start_frames:
#                         video.write(composite(blink_base, buffer_mouth, blink_eyes))
#                 else:
#                     if current_frame_count>skip_start_frames:
#                         video.write(buffer)
#                 cumulative_frame_count += 1
#                 current_frame_count += 1
                
#     video.release()

#     command = f'{FFMPEG_PATH}/ffmpeg -i {SAVE_ROOT}/{filename}_no_audio.{ext} -i {ALIGNER_ROOT}/inputs/{filename}/{filename}.wav -map 0:v:0 -map 1:a:0 -c:v copy -framerate {60}/1 {SAVE_ROOT}/{filename}.{ext}'
#     os.system(f'rm {SAVE_ROOT}/{filename}.{ext}')
#     os.system(command)
    
#     print()
#     print()
#     print(f'Video saved --> {SAVE_ROOT}/{filename}.{ext}')

CHARACTER ANIMATION ALL

In [ ]:
# characters = ['0a4a8a95ac934f4e8d7b58561f7913c9__1453198705448541']
characters = sorted(os.listdir(ASSETS_ROOT+'/mouth/'))

crop_face = False
vis_input = False
fps=120

ext = 'mp4'

suffix = ''
if crop_face:
    suffix = '_face'

skip_start_frames = 0
if filename=='babyshark':
    skip_start_frames = 1063

for character_id in tqdm(characters):
    try:
    # if True:
        MOUTH_ROOT = f'{ASSETS_ROOT}/mouth/{character_id}'
        EYES_ROOT = f'{ASSETS_ROOT}/eyes/{character_id}'
        SAVE_ROOT = f'{VIDEO_SAVE_DIR}/{character_id}/'
        os.makedirs(SAVE_ROOT, exist_ok=True)

        asset_dict = {}
        asset_dict['inpainted_mouth_only'] = cv2.imread(f'{MOUTH_ROOT}/metadata/inpainted_mouth_only.png')
        asset_dict['inpainted_eyes_only'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_eyes_only.png')
        asset_dict['inpainted_eyes_mouth'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_eyes_mouth.png')
        asset_dict['inpainted_face_mouth_only'] = cv2.imread(f'{MOUTH_ROOT}/metadata/inpainted_face_mouth_only.png')
        asset_dict['inpainted_face_eyes_only'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_face_eyes_only.png')
        asset_dict['inpainted_face_eyes_mouth'] = cv2.imread(f'{EYES_ROOT}/metadata/inpainted_face_eyes_mouth.png')

        image_ref = cv2.imread(f'{EYES_ROOT}/metadata/image_face.png')

        w0, h0, _ = cv2.imread(f'{MOUTH_ROOT}/assets/mouth_0{suffix}.png').shape
        print(w0, h0)
        
        mouth_files = os.listdir(f'{MOUTH_ROOT}/assets/')
        for m in mouth_files:
            asset_dict[m[:-4]] = cv2.imread(f'{MOUTH_ROOT}/assets/{m}', -1)

        eye_files = os.listdir(f'{EYES_ROOT}/assets/')
        for e in eye_files:
            asset_dict[e[:-4]] = cv2.imread(f'{EYES_ROOT}/assets/{e}', -1)

        asset_dict.keys()
        
        base = asset_dict[f'inpainted{suffix}_mouth_only']
        base = cv2.resize(base, (h0,w0))

        eyes_id = 0
        blink_id = 2
        blink_gap = 2 #seconds
        num_blink_frames = 12
        
        USE_DEFAULT_MOUTH = False
        USE_DEFAULT_EYES = True
        
        video=cv2.VideoWriter(f'{SAVE_ROOT}/{filename}_no_audio.{ext}',cv2.VideoWriter_fourcc(*'mp4v'),fps,(h0,w0))
        if vis_input:
            video=cv2.VideoWriter(f'{SAVE_ROOT}/{filename}_no_audio.{ext}',cv2.VideoWriter_fourcc(*'mp4v'),fps,(h0,2*w0))
        buffer = np.ones((w0,h0,3)).astype('uint8')*255
        current_frame_count = 0
        
        for i in tqdm(range(len(words))):
            total_duration = words[i].duration()
            word_frame_count = int(fps*total_duration)
            cumulative_frame_count = 0
        
            #eyes asset
            try:
                eyes_id = eye_word_mapping[words[i].mark.lower()]
                base = asset_dict[f'inpainted{suffix}_eyes_mouth']
                base = cv2.resize(base, (h0,w0))
                USE_DEFAULT_EYES=False
            except:
                pass
        
            eyes = asset_dict[f'eyes_{eyes_id}{suffix}']
            eyes = cv2.resize(eyes, (h0,w0))


            
            for p in words_phonemes[i]:
                frame_count = math.ceil(fps*p.duration())
        
                #default mouth asset
                mouth_id = 1
                buffer_mouth = asset_dict[f'mouth_{mouth_id}{suffix}']
                buffer_mouth = cv2.resize(buffer_mouth, (h0,w0))

                
                # default
                buffer = composite(base, buffer_mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
                
                viseme_id=''
                if p.mark!='':
                    try:
                        mouth_id = phoneme2viseme[p.mark[:2]]
                        mouth = asset_dict[f'mouth_{mouth_id}{suffix}']
                        mouth = cv2.resize(mouth, (h0,w0))
                        buffer_mouth = mouth
                        buffer = composite(base, mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
                    except:
                        pass

                for _ in range(frame_count):
                    if cumulative_frame_count>=word_frame_count:
                        break
                    #blink
                    if current_frame_count%(fps*blink_gap)<num_blink_frames:
                        blink_eyes = asset_dict[f'eyes_{blink_id}{suffix}']
                        blink_eyes = cv2.resize(blink_eyes, (h0,w0))
                        blink_base = asset_dict[f'inpainted{suffix}_eyes_mouth']
                        blink_base = cv2.resize(blink_base, (h0,w0))
                        if current_frame_count>skip_start_frames:
                            if not vis_input:
                                video.write(composite(blink_base, buffer_mouth, blink_eyes))
                            else:
                                frame_to_write = np.zeros((2*w0,h0,3)).astype('uint8')
                                frame_to_write[:w0,:] = image_ref
                                frame_to_write[w0:,:] = composite(blink_base, buffer_mouth, blink_eyes)
                                video.write(frame_to_write(blink_base, buffer_mouth, blink_eyes))
                    else:
                        if current_frame_count>skip_start_frames:
                            if not vis_input:
                                video.write(buffer)
                            else:
                                frame_to_write = np.zeros((2*w0,h0,3)).astype('uint8')
                                frame_to_write[:w0,:] = image_ref
                                frame_to_write[w0:,:] = buffer
                                video.write(frame_to_write)

                    cumulative_frame_count += 1
                    current_frame_count += 1
                    
        video.release()

        command = f'{FFMPEG_PATH}/ffmpeg -i {SAVE_ROOT}/{filename}_no_audio.{ext} -i {ALIGNER_ROOT}/inputs/{filename}/{filename}.wav -map 0:v:0 -map 1:a:0 -c:v copy -framerate {60}/1 {SAVE_ROOT}/{filename}_def_eyes.{ext}'
        os.system(f'rm {SAVE_ROOT}/{filename}.{ext}')
        os.system(command)
        
        print()
        print()
        print(f'Video saved --> {SAVE_ROOT}/{filename}.{ext}')
    
    except:
        print("Missing Data!")

  0%|          | 0/1 [00:00<?, ?it/s]

1024 1024


100%|██████████| 63/63 [01:12<00:00,  1.15s/it]
rm: cannot remove '/mnt/users_scratch/astitva/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/VIDEO_RESULTS/image//ppt.mp4': No such file or directory
ffmpeg version 5.0 Copyright (c) 2000-2022 the FFmpeg developers
  built with gcc 11 (GCC)
  configuration: --prefix=/mnt/users_scratch/astitva/WORKSPACE/ffmpeg/installation --disable-debug --disable-x86asm
  libavutil      57. 17.100 / 57. 17.100
  libavcodec     59. 18.100 / 59. 18.100
  libavformat    59. 16.100 / 59. 16.100
  libavdevice    59.  4.100 / 59.  4.100
  libavfilter     8. 24.100 /  8. 24.100
  libswscale      6.  4.100 /  6.  4.100
  libswresample   4.  3.100 /  4.  3.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '/mnt/users_scratch/astitva/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/VIDEO_RESULTS/image//ppt_no_audio.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isomiso2mp41
    encoder         : Lavf59.27.100




Video saved --> /mnt/users_scratch/astitva/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/VIDEO_RESULTS/image//ppt.mp4
